# W07 — Content Action Playbook

This notebook turns validated model output into a practical content action playbook. It: 1) ranks items with reason codes, 2) maps archetypes to actions, 3) states intended use and limits, 4) defines human-review rules and no-go cases, 5) specifies monitoring and retrain triggers, and 6) exports the ranked queue and figures to `work/outputs/` for the paper.

Run top-to-bottom before exporting.

## 0. Environment and data load (run first)

Loads model predictions, baseline scores, and feature vector to compose the playbook.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# locate repo root (same heuristic as other notebooks)
ROOT = Path.cwd().resolve()
for candidate in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (candidate / "data").exists() and (candidate / "scripts").exists():
        ROOT = candidate
        break

OUT_DIR = ROOT / 'work' / 'outputs'
FIG_DIR = ROOT / 'work' / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root:', ROOT)
# load available data sources (be robust to column names)
pred_path = ROOT / 'data' / 'processed' / 'model_predictions.csv'
fv_path = ROOT / 'data' / 'processed' / 'refresh_feature_vector.csv'
baseline_path = ROOT / 'data' / 'processed' / 'baseline_refresh_queue.csv'

pred = pd.read_csv(pred_path) if pred_path.exists() else pd.DataFrame()
fv = pd.read_csv(fv_path) if fv_path.exists() else pd.DataFrame()
baseline = pd.read_csv(baseline_path) if baseline_path.exists() else pd.DataFrame()

print('predictions rows:', len(pred))
print('feature vector rows:', len(fv))
print('baseline rows:', len(baseline))

# Heuristic: find a model score column on predictions
score_cols = [c for c in pred.columns if 'score' in c.lower() or 'prob' in c.lower() or 'pred' in c.lower()]
model_score_col = None
for c in ['model_score','score','predicted_score','prob','probability']:
    if c in pred.columns:
        model_score_col = c
        break
if model_score_col is None and len(score_cols):
    model_score_col = score_cols[0]

# common id column heuristics
id_col = None
for c in ['content_id','id','doc_id']:
    if c in pred.columns or c in fv.columns:
        id_col = c
        break

if id_col is None:
    raise RuntimeError('Cannot find a content id column in predictions or feature vector')

# align dataframes
if len(pred):
    preds = pred[[id_col, model_score_col]].rename(columns={model_score_col: 'model_score'})
else:
    preds = pd.DataFrame(columns=[id_col, 'model_score'])

if id_col in fv.columns:
    base = fv.copy()
else:
    base = pd.DataFrame()

if len(baseline):
    if 'content_id' in baseline.columns and 'baseline_refresh_score' in baseline.columns:
        baseline_lookup = baseline.set_index('content_id')['baseline_refresh_score']
    else:
        baseline_lookup = pd.Series(dtype=float)
else:
    baseline_lookup = pd.Series(dtype=float)

# merge available sources
df = base.merge(preds, on=id_col, how='left') if not base.empty else preds.copy()
df['model_score'] = df['model_score'].fillna(0).astype(float)
# attach baseline score if available
if len(baseline_lookup):
    df['baseline_score'] = df[id_col].map(baseline_lookup).fillna(0).astype(float)
else:
    df['baseline_score'] = 0.0

# Example auxiliary column: recency or freshness (if present)
recency_col = None
for c in ['days_since_refresh','age_days','freshness_days','age']:
    if c in df.columns:
        recency_col = c
        break
if recency_col is None:
    # fallback: create a weak uniform recency factor
    df['recency_factor'] = 1.0
else:
    # larger age -> higher priority, so invert and scale
    df['recency_factor'] = (df[recency_col].rank(pct=True)).fillna(0.5)

# preview
df.head().T.to_dict()

Repo root: /home/supriya-devkota/Desktop/FlyRank_Assignment


predictions rows: 30000
feature vector rows: 30000
baseline rows: 30000


{0: {'content_id': 'content_304f48230142',
  'client_id': 'client_f369cb89fc',
  'search_volume': 10.0,
  'competition': 0.67,
  'competition_level': 'HIGH',
  'cpc': 2.05,
  'content_type': 'keyword article',
  'main_intent': 'transactional',
  'word_count': 3221.0,
  'char_count': 20457.0,
  'provider_used': 'unknown',
  'model_used': 'gemini-2.5-flash',
  'impressions_90d': 3803,
  'clicks_90d': 29,
  'pageviews_90d': 22,
  'sessions_90d': 17,
  'users_90d': 16,
  'engaged_sessions_90d': 1,
  'ai_sessions_90d': 0,
  'scroll_events_90d': 1,
  'days_with_impressions': 88,
  'days_with_sessions': 13,
  'impressions_last_30d': 578,
  'clicks_last_30d': 2,
  'sessions_last_30d': 2,
  'impressions_prev_30d': 987,
  'clicks_prev_30d': 13,
  'sessions_prev_30d': 9,
  'content_age_days': 187,
  'age_tier': '181-365',
  'age_tier_order': 5,
  'days_since_last_update': 20,
  'freshness_tier': '0-30',
  'word_count_tier': '2000-3500',
  'char_count_tier': '15000-25000',
  'ctr': 0.76,
  'avg_po

## 1) Ranked actions + reason codes

Generate a ranked queue with an explicit `reason_code` for each item and an `action` recommendation.

In [ ]:
# Configurable weights for composing priority
MODEL_WEIGHT = 0.8
BASELINE_WEIGHT = 0.15
RECENCY_WEIGHT = 0.05

# normalize model_score and baseline_score to [0,1] if not already
def norm_series(s):
    if s.max() <= 1.0 and s.min() >= 0.0:
        return s.fillna(0.0)
    return ((s - s.min()) / (s.max() - s.min())).fillna(0.0) if s.max() > s.min() else s.fillna(0.0)

df['model_score_n'] = norm_series(df['model_score'])
df['baseline_score_n'] = norm_series(df['baseline_score'])
df['recency_n'] = norm_series(df.get('recency_factor', pd.Series(1.0, index=df.index)))

# composite priority
df['priority'] = (MODEL_WEIGHT * df['model_score_n'] + BASELINE_WEIGHT * df['baseline_score_n'] + RECENCY_WEIGHT * df['recency_n'])

# reason codes: simple rule-based explanations (expandable)
def reason_code(row):
    # high model confidence and high recency -> Refresh now
    if row['model_score_n'] >= 0.8 and row['recency_n'] >= 0.6:
        return 'HIGH_MODEL_CONFIDENCE_RECENT'
    # high model confidence but old -> Manual review recommended
    if row['model_score_n'] >= 0.8 and row['recency_n'] < 0.6:
        return 'HIGH_MODEL_CONFIDENCE_OLD'
    # baseline strong but model weak -> Follow baseline
    if row['baseline_score_n'] >= 0.7 and row['model_score_n'] < 0.5:
        return 'BASELINE_STRONG'
    # medium scores -> Human triage
    if 0.4 <= row['priority'] < 0.8:
        return 'MEDIUM_PRIORITY_HUMAN_TRIAGE'
    return 'LOW_PRIORITY'

def action_from_reason(code):
    mapping = {
        'HIGH_MODEL_CONFIDENCE_RECENT': 'Auto-refresh (queue)',
        'HIGH_MODEL_CONFIDENCE_OLD': 'Human-review before refresh',
        'BASELINE_STRONG': 'Follow baseline rule (manual or scheduled)',
        'MEDIUM_PRIORITY_HUMAN_TRIAGE': 'Triage by editor',
        'LOW_PRIORITY': 'No action / monitor'
    }
    return mapping.get(code, 'No action')

df['reason_code'] = df.apply(reason_code, axis=1)
df['action'] = df['reason_code'].map(action_from_reason)

# archetype mapping (example rules)
def archetype(row):
    # example archetypes based on available columns; adapt to real features
    if 'traffic_rank' in row.index and row['traffic_rank'] <= 1000:
        return 'High-traffic evergreen'
    if row['recency_n'] >= 0.8 and row['model_score_n'] < 0.5:
        return 'Recently changed, model-uncertain'
    if row['model_score_n'] >= 0.85:
        return 'Model-high-confidence'
    return 'Long-tail'

df['archetype'] = df.apply(archetype, axis=1)

# final ranking
ranked = df.sort_values('priority', ascending=False).reset_index(drop=True)
ranked_preview = ranked[[id_col, 'priority', 'model_score_n', 'baseline_score_n', 'recency_n', 'reason_code', 'action', 'archetype']].head(20)
print('Top 20 preview:')
print(ranked_preview.to_string(index=False))

Top 20 preview:
          content_id  priority  model_score_n  baseline_score_n  recency_n                  reason_code               action             archetype
content_c8e9d6ab9013  0.933962       0.935912          0.901550        1.0 HIGH_MODEL_CONFIDENCE_RECENT Auto-refresh (queue) Model-high-confidence
content_f986bd514b6e  0.912258       0.946904          0.698234        1.0 HIGH_MODEL_CONFIDENCE_RECENT Auto-refresh (queue) Model-high-confidence
content_5195668f06db  0.891529       0.899469          0.813022        1.0 HIGH_MODEL_CONFIDENCE_RECENT Auto-refresh (queue) Model-high-confidence
content_8ba781dafa55  0.890824       0.896011          0.826769        1.0 HIGH_MODEL_CONFIDENCE_RECENT Auto-refresh (queue) Model-high-confidence
content_7c2869a87415  0.890338       0.939337          0.592457        1.0 HIGH_MODEL_CONFIDENCE_RECENT Auto-refresh (queue) Model-high-confidence
content_72fdb385e810  0.888142       0.883624          0.874952        1.0 HIGH_MODEL_CONFIDENCE_RECEN

## 2) Intended use and limits

Explain how the playbook should be used and the limits to automation.

**Intended use (short):** Use the ranked queue as a decision-support list for editorial refresh prioritization. The queue orders pages by predicted benefit; editors review top items and apply the recommended action.

**Limits:**
- Scores are observational and dependent on the dataset and split used for validation.
- Do not treat the priority as causal proof of business impact.
- Items with sparse signals (low traffic, missing features) should be deprioritized or routed to human triage.
- The heuristics and weights are tunable; validate before changing automation thresholds.

## 3) Human review rules + No-go list

Rules that must be followed and actions that must NOT be automated.

**Human review rules:**
- Auto-refresh only for `HIGH_MODEL_CONFIDENCE_RECENT` items and a bounded daily quota (e.g., top N per client per day).
- `HIGH_MODEL_CONFIDENCE_OLD` items require editor review to confirm topical relevance before refresh.
- `MEDIUM_PRIORITY_HUMAN_TRIAGE` items go to an editor queue with contextual signals and previous-change history.
- Always surface a short rationale (`reason_code`) and top 3 contributing features to the editor UI.

**No-go (never fully automate):**
- Automated removal or demotion of content without human approval.
- Any action that could delete user-generated content or change ranking signals in production without rollout.
- Decisions that affect revenue streams or legal compliance without explicit product and legal sign-off.

## 4) Monitoring and retrain triggers

Define simple monitoring signals and the conditions that should trigger a model review or retrain.

**Monitoring metrics (examples):**
- Precision@50 and Precision@100 on a rolling weekly evaluation set (client-held-out).
- Distribution shift on key features (KS test for `traffic`, `freshness`, `engagement`).
- Positive-rate drift for the label.

**Retrain / review triggers:**
- Precision@50 drops by >10% relative to baseline window.
- KS-statistic > 0.2 for any major feature vs validation baseline.
- Systematic vendor or product changes (content schema changes) detected.

When triggered: assemble a small review (data engineer + editor + modeling lead), investigate root cause, and run a limited revalidation before redeploying any model changes.

## 5) Exports for the paper

Write the ranked queue CSV and supporting JSON metrics and save any figures that the paper will reuse.

In [ ]:
# Export ranked queue (CSV) and metrics JSON, and draw a small figure.
export_cols = [id_col, 'priority', 'model_score', 'baseline_score', 'reason_code', 'action', 'archetype']
export_df = ranked[export_cols].rename(columns={id_col: 'content_id'})
csv_path = OUT_DIR / 'ranked_refresh_queue.csv'
export_df.to_csv(csv_path, index=False)
print('Wrote ranked queue to', csv_path)

# metrics JSON: simple receipts for paper
metrics = {
    'rows_exported': int(len(export_df)),
    'top1_priority': float(export_df['priority'].iloc[0]) if len(export_df) else None,
    'model_weight': MODEL_WEIGHT,
    'baseline_weight': BASELINE_WEIGHT,
}
json_path = OUT_DIR / 'action_playbook_metrics.json'
with open(json_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print('Wrote metrics to', json_path)

# simple figure: priority distribution (save to figures dir)
plt.figure(figsize=(6,3))
plt.hist(ranked['priority'].dropna(), bins=40, color='#2a9d8f')
plt.title('Priority score distribution')
plt.xlabel('Priority')
plt.ylabel('Count')
fig_path = FIG_DIR / 'priority_distribution.png'
plt.tight_layout()
plt.savefig(fig_path, dpi=150)
plt.close()
print('Wrote figure to', fig_path)

Wrote ranked queue to /home/supriya-devkota/Desktop/FlyRank_Assignment/work/outputs/ranked_refresh_queue.csv
Wrote metrics to /home/supriya-devkota/Desktop/FlyRank_Assignment/work/outputs/action_playbook_metrics.json


Wrote figure to /home/supriya-devkota/Desktop/FlyRank_Assignment/work/figures/priority_distribution.png


## 6) Self-check

Checklist before submission.

- [ ] Notebook runs top-to-bottom with no errors.
- [ ] `work/outputs/ranked_refresh_queue.csv` is produced by running this notebook.
- [ ] Figures used in the paper are in `work/figures/`.
- [ ] Honest description of intended use, limits, human-review rules, monitoring triggers included above.

---
If you want, I can run this notebook here and commit results; would you like me to run it now?